### SAVE AE33 DATA TO DAILY FILES

Pythons script to establish TCP connection to AE33 instrument and save data to daily files.

User has to change the IP address of the instrument and the date range: variables **INSTRUMENT_IP**, **START_DATE** and **END_DATE**.
User has to give the location of the working directory for saving data files: variable **WORKING_DIR**.

Script will create subfolders for each instrument and for each month and save data files with the following name:.<br>
*WORKING_DIR/serial-number/YYYY-MM/serial-number_YYYY-MM-DD.csv*.

Script is prepared to work with multiple instruments - create a list of IP addresses to ***INSTRUMENT_IP***.

**License:** Aerosol Magee Scientific Software License

See LICENSE file for full terms

In [1]:
# IP of the instrument - change it to the actual IP address of your AE33 instrument
# if you want to download data from multiple instruments, you can use a list of IP addresses
INSTRUMENT_IP = ['10.10.10.61', '10.10.10.226', '10.10.10.0', '10.10.10.224']
# INSTRUMENT_IP = '10.10.10.224'

# start and end date of the data to be collected
START_DATE = '2026-07-04'
END_DATE = '2026-07-08'

# working directory for saving data files - if directory does not exist, it will be created
WORKING_DIR = r'C:\Users\mivancic\OneDrive - Aerosol d.o.o\tests\2026-07-06_AE33_test_csv_export'

#### End of a user input section.

In [2]:
from io import StringIO
import re
import socket
import os

import pandas as pd

from aerosol_magee_pytools.data_access.tcp_ip import request_tcp

In [3]:
# default parameters:
INSTRUMENT_PORT = 8002  # always the same
BUFFER_SIZE = 4096  # always the same
RECV_TIMEOUT = 2.0  # seconds; if connection is weak, increase this value

DATETIME_FORMAT = "%m/%d/%Y %I:%M:%S %p"

COLUMNS_AE33_DATA = ['SerialNumber', 'ID', 'StartTime', 'EndTime', 'SetupID', 'SetupTimestamp', 'Ref1', 'Sens11', 'Sens12', 'Ref2', 'Sens21', 'Sens22', 'Ref3', 'Sens31', 'Sens32', 'Ref4', 'Sens41', 'Sens42', 'Ref5', 'Sens51', 'Sens52', 'Ref6', 'Sens61', 'Sens62', 'Ref7', 'Sens71', 'Sens72', 'BC11', 'BC12', 'BC1', 'BC21', 'BC22', 'BC2', 'BC31', 'BC32', 'BC3', 'BC41', 'BC42', 'BC4', 'BC51', 'BC52', 'BC5', 'BC61', 'BC62', 'BC6', 'BC71', 'BC72', 'BC7', 'K1', 'K2', 'K3', 'K4', 'K5', 'K6', 'K7', 'BB', 'Pressure', 'Temp', 'Flow1', 'Flow2', 'FlowC', 'T_controller', 'T_supply', 'T_LED', 'ControllerStatus', 'LEDStatus', 'DetectorStatus', 'ValveStatus', 'Status', 'TapeAdvanceCount', 'TapeAdvanceLeft', 'CPU', 'DiskSpace', 'NumConnections']

COLUMNS_AE33_DATE_ORDER = ['ID', 'Date', 'StartID', 'EndID']

In [4]:
def get_serial_number_ae33(instrument_ip,
                           instrument_port=INSTRUMENT_PORT,
                           buffer_size=BUFFER_SIZE,
                           timeout=RECV_TIMEOUT,
                           verbose=False):

    # use command HELLO to get the serial number
    _command_ae33 = 'HELLO\r\n'
    _data = request_tcp(ip=instrument_ip,
                        port=instrument_port,
                        command=_command_ae33,
                        buffer_size=buffer_size,
                        timeout=timeout)

    if _data is None:
        return None

    if verbose:
        print(_data)

    # parse serial number from the text
    serial_pattern = r'Instrument serialnumber: (AE33-[A-Za-z0-9]+-[A-Za-z0-9]+)'

    # Search for the serial number in the text
    _match = re.search(serial_pattern, _data)
    if _match:
        _serial_number = _match.group(1)
        return _serial_number
    else:
        return None

def get_date_order_ae33(instrument_ip,
                        instrument_port=INSTRUMENT_PORT,
                        buffer_size=BUFFER_SIZE,
                        timeout=RECV_TIMEOUT,
                        verbose=False):

    _command_ae33 = f'FETCH DateOrder 1\r\n'
    _data = request_tcp(ip=instrument_ip,
                        port=instrument_port,
                        command=_command_ae33,
                        buffer_size=buffer_size,
                        timeout=timeout)

    # in the end, you can parse data, for example convert it into pandas dataframe
    _df_ae33_date_order = pd.read_csv(StringIO(_data.replace('AE33>', '').strip()),
                                      sep='|',
                                      names=COLUMNS_AE33_DATE_ORDER)
    _df_ae33_date_order['Date'] = pd.to_datetime(_df_ae33_date_order['Date'],
                                                 format=DATETIME_FORMAT)
    if verbose:
        print()
        print(_df_ae33_date_order)
    return _df_ae33_date_order

def get_ae33_data(instrument_ip, start_id, end_id,
                  instrument_port=INSTRUMENT_PORT,
                  buffer_size=BUFFER_SIZE,
                  timeout=RECV_TIMEOUT,
                  verbose=False):

    _command_ae33 = f'FETCH Data {start_id} {end_id}\r\n'
    _data = request_tcp(ip=instrument_ip,
                        port=instrument_port,
                        command=_command_ae33,
                        buffer_size=buffer_size,
                        timeout=timeout)

    # in the end, you can parse data, for example convert it into pandas dataframe
    _df_ae33_device_data = pd.read_csv(StringIO(_data.replace('AE33>', '').strip()),
                                       sep='|',
                                       names=COLUMNS_AE33_DATA)

    _df_ae33_device_data['StartTime'] = pd.to_datetime(_df_ae33_device_data['StartTime'],
                                                       format=DATETIME_FORMAT)
    _df_ae33_device_data['EndTime'] = pd.to_datetime(_df_ae33_device_data['EndTime'],
                                                     format=DATETIME_FORMAT)
    _df_ae33_device_data['SetupTimestamp'] = pd.to_datetime(_df_ae33_device_data['SetupTimestamp'],
                                                            format=DATETIME_FORMAT)
    if verbose:
        print()
        print(_df_ae33_device_data)
    return _df_ae33_device_data

In [5]:
if type(INSTRUMENT_IP) is str:
    INSTRUMENT_IP = [INSTRUMENT_IP]

# for loop over all instruments
for instr_ip in INSTRUMENT_IP:
    # get instrument serial number
    serial_number = get_serial_number_ae33(instrument_ip=instr_ip)
    if serial_number is None:
        print(f"!!! Error: Could not connect to the device with IP address {instr_ip}. "
              f"The request timed out while waiting for a response.")
        print()
        continue
    print(f"Instrument: {serial_number} ({instr_ip})")

    # get DateOrder table to get data ids
    df_date_order = get_date_order_ae33(instrument_ip=instr_ip)

    # for loop over all days inside start and end time
    for day in pd.date_range(start=pd.to_datetime(START_DATE).normalize(),
                             end=pd.to_datetime(END_DATE).normalize(),
                             freq='D',
                             inclusive='both'):

        daily_ids = df_date_order[df_date_order['Date'] == pd.to_datetime(day).normalize()]
        if not daily_ids.empty:
            print(f"    {serial_number} ({instr_ip}) Date: {day.date()}, "
                  f"StartID: {daily_ids['StartID'].iloc[0]}, EndID: {daily_ids['EndID'].iloc[0]}")

            # get data for the day
            df_device_data = get_ae33_data(instrument_ip=instr_ip,
                                           start_id=daily_ids['StartID'].iloc[0],
                                           end_id=daily_ids['EndID'].iloc[0])

            # prepare subfolder name - {serial}/{year}-{month} and if not exists, create it
            subfolder = os.path.join(WORKING_DIR,
                                     serial_number,
                                     f"{day.year}-{day.month:02d}")
            if not os.path.exists(subfolder):
                os.makedirs(subfolder)

            # prepare filename - {serial}_{year}-{month}-{day}.csv
            filename = os.path.join(subfolder,
                                    f"{serial_number}_{day.year}-{day.month:02d}-{day.day:02d}.csv")

            # save daily data to file
            df_device_data.to_csv(filename, index=False)

        else:
            print(f"    {serial_number} ({instr_ip}) Date: {day.date()}: No data available")

    print()

!!! Error: Could not connect to the device with IP address 10.10.10.61. The request timed out while waiting for a response.

Instrument: AE33-S08-00951 (10.10.10.226)
    AE33-S08-00951 (10.10.10.226) Date: 2026-07-04, StartID: 534060, EndID: 535499
    AE33-S08-00951 (10.10.10.226) Date: 2026-07-05, StartID: 535500, EndID: 536939
    AE33-S08-00951 (10.10.10.226) Date: 2026-07-06, StartID: 536940, EndID: 537720
    AE33-S08-00951 (10.10.10.226) Date: 2026-07-07: No data available
    AE33-S08-00951 (10.10.10.226) Date: 2026-07-08: No data available

!!! Error: Could not connect to the device with IP address 10.10.10.0. The request timed out while waiting for a response.

Instrument: AE33-S06-00565 (10.10.10.224)
    AE33-S06-00565 (10.10.10.224) Date: 2026-07-04, StartID: 3192177, EndID: 3193616
    AE33-S06-00565 (10.10.10.224) Date: 2026-07-05, StartID: 3193617, EndID: 3195056
    AE33-S06-00565 (10.10.10.224) Date: 2026-07-06, StartID: 3195057, EndID: 3195838
    AE33-S06-00565 (10